# immunograph — Demo Pipeline

This notebook walks through the full immunograph pipeline:

1. Load a small immune pathway dataset (NF-κB, JAK-STAT)
2. Build the immune knowledge graph
3. Train node embeddings (Node2Vec baseline)
4. Run causal signal propagation
5. Rank upstream intervention targets
6. Suggest perturbation experiments


In [ ]:
import sys, os
# Add project root to path so imports work from the notebook
sys.path.insert(0, os.path.abspath('..'))

## 1 · Define a Small Sample Dataset

We use a hand-curated set of well-known NF-κB and JAK-STAT interactions
as our MVP dataset.  This avoids any network calls for the demo.

In [ ]:
# Curated NF-κB and JAK-STAT interactions
# Format: (source, target, edge_type, metadata)

SAMPLE_EDGES = [
    # JAK-STAT pathway (IL-6 axis)
    ("IL6",   "IL6R",  "binds",     {"confidence_score": 0.95, "source_publication": "KEGG:hsa04630"}),
    ("IL6R",  "JAK1",  "activates", {"confidence_score": 0.90, "source_publication": "KEGG:hsa04630"}),
    ("JAK1",  "STAT3", "activates", {"confidence_score": 0.92, "source_publication": "KEGG:hsa04630"}),
    ("JAK1",  "STAT1", "activates", {"confidence_score": 0.88, "source_publication": "KEGG:hsa04630"}),
    ("STAT3", "IL6",   "activates", {"confidence_score": 0.75, "source_publication": "PMID:12345678"}),  # feedback
    ("SOCS3", "JAK1",  "inhibits",  {"confidence_score": 0.85, "source_publication": "KEGG:hsa04630"}),

    # NF-κB pathway
    ("TNF",   "TNFR1", "binds",     {"confidence_score": 0.97, "source_publication": "KEGG:hsa04064"}),
    ("TNFR1", "TRAF2", "activates", {"confidence_score": 0.87, "source_publication": "KEGG:hsa04064"}),
    ("TRAF2", "IKK",   "activates", {"confidence_score": 0.85, "source_publication": "KEGG:hsa04064"}),
    ("IKK",   "NFKB1", "activates", {"confidence_score": 0.90, "source_publication": "KEGG:hsa04064"}),
    ("NFKB1", "TNF",   "activates", {"confidence_score": 0.80, "source_publication": "PMID:87654321"}),  # feedback
    ("NFKB1", "IL6",   "activates", {"confidence_score": 0.82, "source_publication": "PMID:11223344"}),
    ("IKBA",  "NFKB1", "inhibits",  {"confidence_score": 0.91, "source_publication": "KEGG:hsa04064"}),

    # Cross-talk
    ("STAT3", "NFKB1", "activates", {"confidence_score": 0.70, "source_publication": "PMID:99887766"}),
    ("NFKB1", "STAT3", "activates", {"confidence_score": 0.68, "source_publication": "PMID:55443322"}),
]

NODE_TYPE_MAP = {
    "IL6":   "Cytokine",
    "IL6R":  "Receptor",
    "JAK1":  "Protein",
    "STAT3": "TranscriptionFactor",
    "STAT1": "TranscriptionFactor",
    "SOCS3": "Protein",
    "TNF":   "Cytokine",
    "TNFR1": "Receptor",
    "TRAF2": "Protein",
    "IKK":   "Protein",
    "NFKB1": "TranscriptionFactor",
    "IKBA":  "Protein",
}

print(f"Sample dataset: {len(SAMPLE_EDGES)} edges, {len(NODE_TYPE_MAP)} nodes")

## 2 · Build the Immune Knowledge Graph

In [ ]:
from graph.graph_builder import ImmuneGraphBuilder
from graph.graph_queries import GraphQueryEngine

builder = ImmuneGraphBuilder(use_memory_backend=True)
builder.load_edge_list(SAMPLE_EDGES, node_type_map=NODE_TYPE_MAP)

query_engine = GraphQueryEngine(builder)

print("Graph built successfully.")
print(f"  Nodes : {len(builder.get_nodes())}")
print(f"  Edges : {len(builder.get_edge_list())}")

In [ ]:
# Explore neighbourhood of NFKB1
nfkb_neighbours = query_engine.get_neighbours('NFKB1', direction='both')
print("NFKB1 neighbours:", nfkb_neighbours)

# Top hub nodes
hubs = query_engine.top_hubs(n=5)
print("Top 5 hubs:", hubs)

## 3 · Generate Node Embeddings (Node2Vec Baseline)

In [ ]:
from embeddings.node2vec_baseline import Node2VecBaseline
from embeddings.inference import EmbeddingInference

node_ids = list(builder.get_nodes().keys())

n2v = Node2VecBaseline(
    embedding_dim=16,
    walk_length=10,
    num_walks=5,
    window_size=3,
    epochs=10,
    random_seed=42,
)
n2v.fit(node_ids, SAMPLE_EDGES)

inference = EmbeddingInference(n2v)

print("Embeddings trained.  Vocabulary:", inference.vocabulary)

In [ ]:
# Most similar nodes to IL6
similar = inference.most_similar('IL6', top_k=5)
print("Nodes most similar to IL6:")
for item in similar:
    print(f"  {item['node_id']:10s}  sim={item['similarity']:.4f}")

In [ ]:
# JAK-STAT pathway subgraph embedding
jak_stat_nodes = ['IL6', 'IL6R', 'JAK1', 'STAT3', 'SOCS3']
jak_stat_emb = inference.get_subgraph_embedding(jak_stat_nodes)
print(f"JAK-STAT subgraph embedding shape: {jak_stat_emb.shape}")
print(f"First 5 dims: {jak_stat_emb[:5]}")

## 4 · Causal Signal Propagation

In [ ]:
from causal.propagation import CausalPropagator

propagator = CausalPropagator(decay=0.8, max_iterations=30)

# Seed signal at TNF and propagate downstream
seed_scores = {'TNF': 1.0}
downstream_scores = propagator.propagate(seed_scores, SAMPLE_EDGES, direction='downstream')

print("Downstream propagation from TNF:")
for node, score in sorted(downstream_scores.items(), key=lambda x: -abs(x[1])):
    if node != 'TNF':
        print(f"  {node:10s}  score={score:+.4f}")

## 5 · Rank Intervention Targets

In [ ]:
from causal.intervention_ranker import InterventionRanker

ranker = InterventionRanker(propagator=propagator)

# Rank nodes that could suppress IL6 (a key pro-inflammatory cytokine)
rankings = ranker.rank_interventions('IL6', SAMPLE_EDGES, top_k=8)

print("Top intervention candidates to modulate IL6:")
print(f"  {'Rank':<5} {'Node':12} {'Score':>8} {'Propagation':>12} {'Structural':>11}")
print("  " + "-" * 52)
for i, r in enumerate(rankings, 1):
    print(f"  {i:<5} {r['node_id']:12} {r['score']:8.4f} {r['propagation_score']:12.4f} {r['structural_score']:11.4f}")

## 6 · Suggest Perturbation Experiments

In [ ]:
from optimizer.experiment_suggester import ExperimentSuggester

suggester = ExperimentSuggester(ranker=ranker)

suggestions = suggester.suggest_experiments(
    objective='IL6',
    edge_list=SAMPLE_EDGES,
    budget=3,
)

print("Suggested perturbation experiments to reduce IL6 signaling:\n")
for s in suggestions:
    print(f"  Experiment #{s['rank']}: Perturb {s['node_id']}")
    print(f"    Causal score    : {s['causal_score']:.4f}")
    print(f"    Rationale       : {s['rationale']}")
    print()

## 7 · Simulate an Experimental Observation

After running an experiment, feed the result back into the active learner
to update its belief state and get improved suggestions for the next round.

In [ ]:
# Simulate: perturbing NFKB1 produced a log-fold-change of -1.8 in IL6
if suggestions:
    top_node = suggestions[0]['node_id']
    suggester.record_result(top_node, observed_effect=-1.8)
    print(f"Recorded observation for '{top_node}': LFC = -1.8")

# Get updated suggestions excluding the already-tested node
updated_suggestions = suggester.suggest_experiments(
    objective='IL6',
    edge_list=SAMPLE_EDGES,
    budget=3,
    exclude=[top_node] if suggestions else [],
)

print("\nUpdated suggestions (excluding already-tested nodes):")
for s in updated_suggestions:
    print(f"  #{s['rank']}: {s['node_id']}  (causal={s['causal_score']:.4f})")

---

## Summary

| Step | Tool | Output |
|---|---|---|
| Data ingestion | `SAMPLE_EDGES` (curated) | Edge list |
| Graph construction | `ImmuneGraphBuilder` | In-memory KG |
| Embeddings | `Node2VecBaseline` | 16-dim node vectors |
| Causal propagation | `CausalPropagator` | Signed influence scores |
| Intervention ranking | `InterventionRanker` | Ranked candidate list |
| Experiment suggestion | `ExperimentSuggester` | Prioritised experiment plan |

Next steps:
- Ingest real KEGG pathway data using `pathway_loader.load_kegg_pathway('hsa04630')`
- Expand with PubMed abstracts via `pubmed_parser.search_pubmed('JAK STAT autoimmune')`
- Replace Node2Vec with the `GNNTrainer` for richer structural encodings
- Launch `api/app.py` for REST-based querying